# WEEK 2 : DATA CLEANING & FEATURE ENGINEERING

Import Libraries

In [184]:
import os
import pandas as pd
import numpy as np

# Set Display Options
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Define File Paths
IN_PATH = os.path.join("..", "data", "processed", "week1_sample.csv")
OUT_PATH = os.path.join("..", "data", "processed", "week2_processed.csv")

print("Input path:", IN_PATH)
print("Output path:", OUT_PATH)
print("Input file exists?", os.path.exists(IN_PATH))

Input path: ..\data\processed\week1_sample.csv
Output path: ..\data\processed\week2_processed.csv
Input file exists? True


Load Dataset

In [185]:
df = pd.read_csv(IN_PATH)
print("Data loaded successfully.")
print("Shape:", df.shape)
df.head(5)


Data loaded successfully.
Shape: (98619, 15)


,Passenger ID,First Name,Last Name,Gender,Age,Nationality,Airport Name,Airport Country Code,Country Name,Airport Continent,Continents,Departure Date,Arrival Airport,Pilot Name,Flight Status
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,6/28/2022,CXF,Fransisco Hazeldine,On Time
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,12/26/2022,YCO,Marla Parsonage,On Time
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,EU,Europe,1/18/2022,GNB,Rhonda Amber,On Time
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,9/16/2022,YND,Kacie Commucci,Delayed
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2/25/2022,SEE,Ebonee Tree,On Time


Clean Column Names

In [186]:
df.columns = [c.strip().replace(" ", "_") for c in df.columns]
print("Column names cleaned")
df.columns.tolist()[:30]


Column names cleaned


['Passenger_ID',
 'First_Name',
 'Last_Name',
 'Gender',
 'Age',
 'Nationality',
 'Airport_Name',
 'Airport_Country_Code',
 'Country_Name',
 'Airport_Continent',
 'Continents',
 'Departure_Date',
 'Arrival_Airport',
 'Pilot_Name',
 'Flight_Status']

Handle Missing Values

In [187]:
missing_before = df.isna().sum().sort_values(ascending=False)
missing_before.head(15)


Passenger_ID            0
First_Name              0
Last_Name               0
Gender                  0
Age                     0
Nationality             0
Airport_Name            0
Airport_Country_Code    0
Country_Name            0
Airport_Continent       0
Continents              0
Departure_Date          0
Arrival_Airport         0
Pilot_Name              0
Flight_Status           0
dtype: int64

Fill missing values

In [188]:
num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = df.select_dtypes(exclude="number").columns.tolist()

for c in num_cols:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].median())

for c in cat_cols:
    if df[c].isna().any():
        df[c] = df[c].fillna("Unknown")

print("Missing values handled/updated (numeric=median, text='Unknown')")


Missing values handled/updated (numeric=median, text='Unknown')


In [189]:
missing_after = df.isna().sum().sort_values(ascending=False)
missing_after.head(15)


Passenger_ID            0
First_Name              0
Last_Name               0
Gender                  0
Age                     0
Nationality             0
Airport_Name            0
Airport_Country_Code    0
Country_Name            0
Airport_Continent       0
Continents              0
Departure_Date          0
Arrival_Airport         0
Pilot_Name              0
Flight_Status           0
dtype: int64

Convert Date Column

In [190]:
df["Departure_Date"] = pd.to_datetime(df["Departure_Date"], errors="coerce")
print("Departure_Date converted to datetime")
df["Departure_Date"].head()


Departure_Date converted to datetime


0   2022-06-28
1   2022-12-26
2   2022-01-18
3   2022-09-16
4   2022-02-25
Name: Departure_Date, dtype: datetime64[ns]

CLEAN COUNTRY NAMES

In [191]:



df["Country_Name"] = (
    df["Country_Name"]
    .astype(str)
    .str.strip()
    .str.title()
)

country_fix = {
    "Usa": "United States",
    "United States Of America": "United States",
    "U.S.A": "United States",
    "Uk": "United Kingdom",
    "Uae": "United Arab Emirates"
}

df["Country_Name"] = df["Country_Name"].replace(country_fix)

In [192]:
continent_map = {
    "AS": "Asia",
    "EU": "Europe",
    "NA": "North America",
    "SA": "South America",
    "AF": "Africa",
    "OC": "Oceania"
}

df["Airport_Continent"] = df["Airport_Continent"].replace(continent_map)

In [193]:
df.head(5)

,Passenger_ID,First_Name,Last_Name,Gender,Age,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Departure_Date,Arrival_Airport,Pilot_Name,Flight_Status
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,2022-06-28,CXF,Fransisco Hazeldine,On Time
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,2022-12-26,YCO,Marla Parsonage,On Time
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,2022-01-18,GNB,Rhonda Amber,On Time
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,2022-09-16,YND,Kacie Commucci,Delayed
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2022-02-25,SEE,Ebonee Tree,On Time


Create Date Features

In [194]:
df["Departure_Year"] = df["Departure_Date"].dt.year
df["Departure_Month"] = df["Departure_Date"].dt.month
df["Departure_Day"] = df["Departure_Date"].dt.day
df["Departure_DayOfWeek"] = df["Departure_Date"].dt.day_name()

print("Time-based features created:")
df[["Departure_Year", "Departure_Month", "Departure_Day", "Departure_DayOfWeek"]].head()


Time-based features created:


,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek
0,2022.0,6.0,28.0,Tuesday
1,2022.0,12.0,26.0,Monday
2,2022.0,1.0,18.0,Tuesday
3,2022.0,9.0,16.0,Friday
4,2022.0,2.0,25.0,Friday


MONTH NAME COLUMN

In [195]:
month_map = {
1:"Jan",2:"Feb",3:"Mar",4:"Apr",
5:"May",6:"Jun",7:"Jul",8:"Aug",
9:"Sep",10:"Oct",11:"Nov",12:"Dec"
}

df["Month_Name"] = df["Departure_Month"].map(month_map)

In [196]:
month_order = [
"Jan","Feb","Mar","Apr","May","Jun",
"Jul","Aug","Sep","Oct","Nov","Dec"
]

df["Month_Name"] = pd.Categorical(
df["Month_Name"],
categories=month_order,
ordered=True
)

In [197]:
df.head(5)

,Passenger_ID,First_Name,Last_Name,Gender,Age,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Departure_Date,Arrival_Airport,Pilot_Name,Flight_Status,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek,Month_Name
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,2022-06-28,CXF,Fransisco Hazeldine,On Time,2022.0,6.0,28.0,Tuesday,Jun
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,2022-12-26,YCO,Marla Parsonage,On Time,2022.0,12.0,26.0,Monday,Dec
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,2022-01-18,GNB,Rhonda Amber,On Time,2022.0,1.0,18.0,Tuesday,Jan
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,2022-09-16,YND,Kacie Commucci,Delayed,2022.0,9.0,16.0,Friday,Sep
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2022-02-25,SEE,Ebonee Tree,On Time,2022.0,2.0,25.0,Friday,Feb


Correct Data Type for Date Features

In [198]:

for c in ["Departure_Year", "Departure_Month", "Departure_Day"]:
    if c in df.columns:
        df[c] = df[c].astype("Int64")  # keeps missing values safe

df[["Departure_Year", "Departure_Month", "Departure_Day", "Departure_DayOfWeek"]].head()


,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek
0,2022,6,28,Tuesday
1,2022,12,26,Monday
2,2022,1,18,Tuesday
3,2022,9,16,Friday
4,2022,2,25,Friday


In [199]:
weekday_order = [
"Monday","Tuesday","Wednesday",
"Thursday","Friday","Saturday","Sunday"
]

df["Departure_DayOfWeek"] = pd.Categorical(
df["Departure_DayOfWeek"],
categories=weekday_order,
ordered=True
)

In [200]:
df.head(5)


,Passenger_ID,First_Name,Last_Name,Gender,Age,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Departure_Date,Arrival_Airport,Pilot_Name,Flight_Status,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek,Month_Name
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,2022-06-28,CXF,Fransisco Hazeldine,On Time,2022,6,28,Tuesday,Jun
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,2022-12-26,YCO,Marla Parsonage,On Time,2022,12,26,Monday,Dec
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,2022-01-18,GNB,Rhonda Amber,On Time,2022,1,18,Tuesday,Jan
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,2022-09-16,YND,Kacie Commucci,Delayed,2022,9,16,Friday,Sep
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2022-02-25,SEE,Ebonee Tree,On Time,2022,2,25,Friday,Feb


Create Age Groups

In [201]:
df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[0, 12, 19, 30, 55, 100],
    labels=["Child", "Teen", "Youth", "Adult", "Senior Citizen"]
)

print("Age_Group feature created")
df[["Age", "Age_Group"]].head(10)


Age_Group feature created


,Age,Age_Group
0,62,Senior Citizen
1,62,Senior Citizen
2,67,Senior Citizen
3,71,Senior Citizen
4,21,Youth
5,55,Adult
6,73,Senior Citizen
7,36,Adult
8,35,Adult
9,13,Teen


In [202]:
df.head(5)

,Passenger_ID,First_Name,Last_Name,Gender,Age,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Departure_Date,Arrival_Airport,Pilot_Name,Flight_Status,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek,Month_Name,Age_Group
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,2022-06-28,CXF,Fransisco Hazeldine,On Time,2022,6,28,Tuesday,Jun,Senior Citizen
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,2022-12-26,YCO,Marla Parsonage,On Time,2022,12,26,Monday,Dec,Senior Citizen
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,2022-01-18,GNB,Rhonda Amber,On Time,2022,1,18,Tuesday,Jan,Senior Citizen
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,2022-09-16,YND,Kacie Commucci,Delayed,2022,9,16,Friday,Sep,Senior Citizen
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2022-02-25,SEE,Ebonee Tree,On Time,2022,2,25,Friday,Feb,Youth


Create Full Name

In [203]:
df["Passenger_Full_Name"] = df["First_Name"] + " " + df["Last_Name"]
print("Passenger_Full_Name created")
df[["Passenger_Full_Name"]].head()


Passenger_Full_Name created


,Passenger_Full_Name
0,Edithe Leggis
1,Elwood Catt
2,Darby Felgate
3,Dominica Pyle
4,Bay Pencost


In [204]:
df.head(5)

,Passenger_ID,First_Name,Last_Name,Gender,Age,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Departure_Date,Arrival_Airport,Pilot_Name,Flight_Status,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek,Month_Name,Age_Group,Passenger_Full_Name
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,2022-06-28,CXF,Fransisco Hazeldine,On Time,2022,6,28,Tuesday,Jun,Senior Citizen,Edithe Leggis
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,2022-12-26,YCO,Marla Parsonage,On Time,2022,12,26,Monday,Dec,Senior Citizen,Elwood Catt
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,2022-01-18,GNB,Rhonda Amber,On Time,2022,1,18,Tuesday,Jan,Senior Citizen,Darby Felgate
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,2022-09-16,YND,Kacie Commucci,Delayed,2022,9,16,Friday,Sep,Senior Citizen,Dominica Pyle
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2022-02-25,SEE,Ebonee Tree,On Time,2022,2,25,Friday,Feb,Youth,Bay Pencost


Create Delay Flag

In [205]:
df["Is_Flight_Delayed"] = df["Flight_Status"].apply(
    lambda x: 1 if str(x).lower() not in ["on time", "ontime", "scheduled"] else 0
)

print("Is_Flight_Delayed feature created")
df[["Flight_Status", "Is_Flight_Delayed"]].head(10)


Is_Flight_Delayed feature created


,Flight_Status,Is_Flight_Delayed
0,On Time,0
1,On Time,0
2,On Time,0
3,Delayed,1
4,On Time,0
5,On Time,0
6,Cancelled,1
7,Cancelled,1
8,On Time,0
9,On Time,0


In [206]:
df.head(5)

,Passenger_ID,First_Name,Last_Name,Gender,Age,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Departure_Date,Arrival_Airport,Pilot_Name,Flight_Status,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek,Month_Name,Age_Group,Passenger_Full_Name,Is_Flight_Delayed
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,2022-06-28,CXF,Fransisco Hazeldine,On Time,2022,6,28,Tuesday,Jun,Senior Citizen,Edithe Leggis,0
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,2022-12-26,YCO,Marla Parsonage,On Time,2022,12,26,Monday,Dec,Senior Citizen,Elwood Catt,0
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,2022-01-18,GNB,Rhonda Amber,On Time,2022,1,18,Tuesday,Jan,Senior Citizen,Darby Felgate,0
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,2022-09-16,YND,Kacie Commucci,Delayed,2022,9,16,Friday,Sep,Senior Citizen,Dominica Pyle,1
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2022-02-25,SEE,Ebonee Tree,On Time,2022,2,25,Friday,Feb,Youth,Bay Pencost,0


Delay Cause

In [207]:

def classify_delay(status):

    status = str(status).lower()

    if status == "cancelled":
        return "Weather / Climatic"

    elif status == "delayed":
        return "Operational / Technical"

    else:
        return "No Delay"

df["Delay_Cause"] = df["Flight_Status"].apply(classify_delay)

print("Delay_Cause feature created")
df[["Flight_Status","Delay_Cause"]].head()

Delay_Cause feature created


,Flight_Status,Delay_Cause
0,On Time,No Delay
1,On Time,No Delay
2,On Time,No Delay
3,Delayed,Operational / Technical
4,On Time,No Delay


In [208]:
df.head(5)

,Passenger_ID,First_Name,Last_Name,Gender,Age,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Departure_Date,Arrival_Airport,Pilot_Name,Flight_Status,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek,Month_Name,Age_Group,Passenger_Full_Name,Is_Flight_Delayed,Delay_Cause
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,2022-06-28,CXF,Fransisco Hazeldine,On Time,2022,6,28,Tuesday,Jun,Senior Citizen,Edithe Leggis,0,No Delay
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,2022-12-26,YCO,Marla Parsonage,On Time,2022,12,26,Monday,Dec,Senior Citizen,Elwood Catt,0,No Delay
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,2022-01-18,GNB,Rhonda Amber,On Time,2022,1,18,Tuesday,Jan,Senior Citizen,Darby Felgate,0,No Delay
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,2022-09-16,YND,Kacie Commucci,Delayed,2022,9,16,Friday,Sep,Senior Citizen,Dominica Pyle,1,Operational / Technical
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2022-02-25,SEE,Ebonee Tree,On Time,2022,2,25,Friday,Feb,Youth,Bay Pencost,0,No Delay


Delay Severity Level

In [209]:

df["Delay_Severity"] = df["Is_Flight_Delayed"].map({
    0:"No Impact",
    1:"Operational Impact"
})

df[["Is_Flight_Delayed","Delay_Severity"]].head()

,Is_Flight_Delayed,Delay_Severity
0,0,No Impact
1,0,No Impact
2,0,No Impact
3,1,Operational Impact
4,0,No Impact


In [210]:
df.head(5)

,Passenger_ID,First_Name,Last_Name,Gender,Age,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Departure_Date,Arrival_Airport,Pilot_Name,Flight_Status,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek,Month_Name,Age_Group,Passenger_Full_Name,Is_Flight_Delayed,Delay_Cause,Delay_Severity
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,2022-06-28,CXF,Fransisco Hazeldine,On Time,2022,6,28,Tuesday,Jun,Senior Citizen,Edithe Leggis,0,No Delay,No Impact
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,2022-12-26,YCO,Marla Parsonage,On Time,2022,12,26,Monday,Dec,Senior Citizen,Elwood Catt,0,No Delay,No Impact
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,2022-01-18,GNB,Rhonda Amber,On Time,2022,1,18,Tuesday,Jan,Senior Citizen,Darby Felgate,0,No Delay,No Impact
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,2022-09-16,YND,Kacie Commucci,Delayed,2022,9,16,Friday,Sep,Senior Citizen,Dominica Pyle,1,Operational / Technical,Operational Impact
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2022-02-25,SEE,Ebonee Tree,On Time,2022,2,25,Friday,Feb,Youth,Bay Pencost,0,No Delay,No Impact


Operational Score

In [211]:

df["Operational_Risk"] = np.where(
    df["Flight_Status"]=="Cancelled",
    "High Risk",
    np.where(
        df["Flight_Status"]=="Delayed",
        "Medium Risk",
        "Low Risk"
    )
)

df[["Flight_Status","Operational_Risk"]].head()

,Flight_Status,Operational_Risk
0,On Time,Low Risk
1,On Time,Low Risk
2,On Time,Low Risk
3,Delayed,Medium Risk
4,On Time,Low Risk


In [212]:
df.head(5)

,Passenger_ID,First_Name,Last_Name,Gender,Age,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Departure_Date,Arrival_Airport,Pilot_Name,Flight_Status,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek,Month_Name,Age_Group,Passenger_Full_Name,Is_Flight_Delayed,Delay_Cause,Delay_Severity,Operational_Risk
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,2022-06-28,CXF,Fransisco Hazeldine,On Time,2022,6,28,Tuesday,Jun,Senior Citizen,Edithe Leggis,0,No Delay,No Impact,Low Risk
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,2022-12-26,YCO,Marla Parsonage,On Time,2022,12,26,Monday,Dec,Senior Citizen,Elwood Catt,0,No Delay,No Impact,Low Risk
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,2022-01-18,GNB,Rhonda Amber,On Time,2022,1,18,Tuesday,Jan,Senior Citizen,Darby Felgate,0,No Delay,No Impact,Low Risk
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,2022-09-16,YND,Kacie Commucci,Delayed,2022,9,16,Friday,Sep,Senior Citizen,Dominica Pyle,1,Operational / Technical,Operational Impact,Medium Risk
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2022-02-25,SEE,Ebonee Tree,On Time,2022,2,25,Friday,Feb,Youth,Bay Pencost,0,No Delay,No Impact,Low Risk


Pilot Workload Feature

In [213]:

pilot_counts = df["Pilot_Name"].value_counts()

df["Pilot_Workload"] = df["Pilot_Name"].map(pilot_counts)

df[["Pilot_Name","Pilot_Workload"]].head()

,Pilot_Name,Pilot_Workload
0,Fransisco Hazeldine,1
1,Marla Parsonage,1
2,Rhonda Amber,1
3,Kacie Commucci,1
4,Ebonee Tree,1


In [214]:
df.head(5)

,Passenger_ID,First_Name,Last_Name,Gender,Age,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Departure_Date,Arrival_Airport,Pilot_Name,Flight_Status,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek,Month_Name,Age_Group,Passenger_Full_Name,Is_Flight_Delayed,Delay_Cause,Delay_Severity,Operational_Risk,Pilot_Workload
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,2022-06-28,CXF,Fransisco Hazeldine,On Time,2022,6,28,Tuesday,Jun,Senior Citizen,Edithe Leggis,0,No Delay,No Impact,Low Risk,1
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,2022-12-26,YCO,Marla Parsonage,On Time,2022,12,26,Monday,Dec,Senior Citizen,Elwood Catt,0,No Delay,No Impact,Low Risk,1
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,2022-01-18,GNB,Rhonda Amber,On Time,2022,1,18,Tuesday,Jan,Senior Citizen,Darby Felgate,0,No Delay,No Impact,Low Risk,1
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,2022-09-16,YND,Kacie Commucci,Delayed,2022,9,16,Friday,Sep,Senior Citizen,Dominica Pyle,1,Operational / Technical,Operational Impact,Medium Risk,1
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2022-02-25,SEE,Ebonee Tree,On Time,2022,2,25,Friday,Feb,Youth,Bay Pencost,0,No Delay,No Impact,Low Risk,1


Monthly Traffic Volume

In [215]:
monthly_counts = df["Departure_Month"].value_counts()

df["Monthly_Traffic"] = df["Departure_Month"].map(monthly_counts)

df[["Departure_Month","Monthly_Traffic"]].head()

,Departure_Month,Monthly_Traffic
0,6,4872
1,12,4732
2,1,5217
3,9,4897
4,2,4398


In [216]:
df.head(5)

,Passenger_ID,First_Name,Last_Name,Gender,Age,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Departure_Date,Arrival_Airport,Pilot_Name,Flight_Status,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek,Month_Name,Age_Group,Passenger_Full_Name,Is_Flight_Delayed,Delay_Cause,Delay_Severity,Operational_Risk,Pilot_Workload,Monthly_Traffic
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,2022-06-28,CXF,Fransisco Hazeldine,On Time,2022,6,28,Tuesday,Jun,Senior Citizen,Edithe Leggis,0,No Delay,No Impact,Low Risk,1,4872
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,2022-12-26,YCO,Marla Parsonage,On Time,2022,12,26,Monday,Dec,Senior Citizen,Elwood Catt,0,No Delay,No Impact,Low Risk,1,4732
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,2022-01-18,GNB,Rhonda Amber,On Time,2022,1,18,Tuesday,Jan,Senior Citizen,Darby Felgate,0,No Delay,No Impact,Low Risk,1,5217
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,2022-09-16,YND,Kacie Commucci,Delayed,2022,9,16,Friday,Sep,Senior Citizen,Dominica Pyle,1,Operational / Technical,Operational Impact,Medium Risk,1,4897
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2022-02-25,SEE,Ebonee Tree,On Time,2022,2,25,Friday,Feb,Youth,Bay Pencost,0,No Delay,No Impact,Low Risk,1,4398


Weekly Traffic Volume

In [217]:
weekday_counts = df["Departure_DayOfWeek"].value_counts()

df["Weekly_Traffic"] = df["Departure_DayOfWeek"].map(weekday_counts)

df[["Departure_DayOfWeek","Weekly_Traffic"]].head()

,Departure_DayOfWeek,Weekly_Traffic
0,Tuesday,8386
1,Monday,8758
2,Tuesday,8386
3,Friday,8462
4,Friday,8462


In [218]:
df.head(5)

,Passenger_ID,First_Name,Last_Name,Gender,Age,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Departure_Date,Arrival_Airport,Pilot_Name,Flight_Status,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek,Month_Name,Age_Group,Passenger_Full_Name,Is_Flight_Delayed,Delay_Cause,Delay_Severity,Operational_Risk,Pilot_Workload,Monthly_Traffic,Weekly_Traffic
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,2022-06-28,CXF,Fransisco Hazeldine,On Time,2022,6,28,Tuesday,Jun,Senior Citizen,Edithe Leggis,0,No Delay,No Impact,Low Risk,1,4872,8386
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,2022-12-26,YCO,Marla Parsonage,On Time,2022,12,26,Monday,Dec,Senior Citizen,Elwood Catt,0,No Delay,No Impact,Low Risk,1,4732,8758
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,2022-01-18,GNB,Rhonda Amber,On Time,2022,1,18,Tuesday,Jan,Senior Citizen,Darby Felgate,0,No Delay,No Impact,Low Risk,1,5217,8386
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,2022-09-16,YND,Kacie Commucci,Delayed,2022,9,16,Friday,Sep,Senior Citizen,Dominica Pyle,1,Operational / Technical,Operational Impact,Medium Risk,1,4897,8462
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2022-02-25,SEE,Ebonee Tree,On Time,2022,2,25,Friday,Feb,Youth,Bay Pencost,0,No Delay,No Impact,Low Risk,1,4398,8462


Airport Congestion Index

In [219]:
airport_counts = df["Airport_Name"].value_counts()

df["Airport_Congestion"] = df["Airport_Name"].map(airport_counts)

df[["Airport_Name","Airport_Congestion"]].head()

,Airport_Name,Airport_Congestion
0,Coldfoot Airport,11
1,Kugluktuk Airport,9
2,Grenoble-Isère Airport,15
3,Ottawa / Gatineau Airport,7
4,Gillespie Field,11


In [220]:
df.head(5)

,Passenger_ID,First_Name,Last_Name,Gender,Age,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Departure_Date,Arrival_Airport,Pilot_Name,Flight_Status,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek,Month_Name,Age_Group,Passenger_Full_Name,Is_Flight_Delayed,Delay_Cause,Delay_Severity,Operational_Risk,Pilot_Workload,Monthly_Traffic,Weekly_Traffic,Airport_Congestion
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,2022-06-28,CXF,Fransisco Hazeldine,On Time,2022,6,28,Tuesday,Jun,Senior Citizen,Edithe Leggis,0,No Delay,No Impact,Low Risk,1,4872,8386,11
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,2022-12-26,YCO,Marla Parsonage,On Time,2022,12,26,Monday,Dec,Senior Citizen,Elwood Catt,0,No Delay,No Impact,Low Risk,1,4732,8758,9
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,2022-01-18,GNB,Rhonda Amber,On Time,2022,1,18,Tuesday,Jan,Senior Citizen,Darby Felgate,0,No Delay,No Impact,Low Risk,1,5217,8386,15
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,2022-09-16,YND,Kacie Commucci,Delayed,2022,9,16,Friday,Sep,Senior Citizen,Dominica Pyle,1,Operational / Technical,Operational Impact,Medium Risk,1,4897,8462,7
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2022-02-25,SEE,Ebonee Tree,On Time,2022,2,25,Friday,Feb,Youth,Bay Pencost,0,No Delay,No Impact,Low Risk,1,4398,8462,11


Create Airport Location

In [221]:
df["Airport_Location"] = df["Airport_Name"] + ", " + df["Country_Name"]
print("Airport_Location feature created")
df[["Airport_Location"]].head()


Airport_Location feature created


,Airport_Location
0,"Coldfoot Airport, United States"
1,"Kugluktuk Airport, Canada"
2,"Grenoble-Isère Airport, France"
3,"Ottawa / Gatineau Airport, Canada"
4,"Gillespie Field, United States"


In [222]:
df.head(5)

,Passenger_ID,First_Name,Last_Name,Gender,Age,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Departure_Date,Arrival_Airport,Pilot_Name,Flight_Status,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek,Month_Name,Age_Group,Passenger_Full_Name,Is_Flight_Delayed,Delay_Cause,Delay_Severity,Operational_Risk,Pilot_Workload,Monthly_Traffic,Weekly_Traffic,Airport_Congestion,Airport_Location
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,2022-06-28,CXF,Fransisco Hazeldine,On Time,2022,6,28,Tuesday,Jun,Senior Citizen,Edithe Leggis,0,No Delay,No Impact,Low Risk,1,4872,8386,11,"Coldfoot Airport, United States"
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,2022-12-26,YCO,Marla Parsonage,On Time,2022,12,26,Monday,Dec,Senior Citizen,Elwood Catt,0,No Delay,No Impact,Low Risk,1,4732,8758,9,"Kugluktuk Airport, Canada"
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,2022-01-18,GNB,Rhonda Amber,On Time,2022,1,18,Tuesday,Jan,Senior Citizen,Darby Felgate,0,No Delay,No Impact,Low Risk,1,5217,8386,15,"Grenoble-Isère Airport, France"
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,2022-09-16,YND,Kacie Commucci,Delayed,2022,9,16,Friday,Sep,Senior Citizen,Dominica Pyle,1,Operational / Technical,Operational Impact,Medium Risk,1,4897,8462,7,"Ottawa / Gatineau Airport, Canada"
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2022-02-25,SEE,Ebonee Tree,On Time,2022,2,25,Friday,Feb,Youth,Bay Pencost,0,No Delay,No Impact,Low Risk,1,4398,8462,11,"Gillespie Field, United States"


Create Flight Type

In [223]:
df["Flight_Type"] = df.apply(
    lambda row: "Domestic" if row["Nationality"] == row["Country_Name"] else "International",
    axis=1
)

print("Flight_Type feature created")
df[["Nationality", "Country_Name", "Flight_Type"]].head(10)


Flight_Type feature created


,Nationality,Country_Name,Flight_Type
0,Japan,United States,International
1,Nicaragua,Canada,International
2,Russia,France,International
3,China,Canada,International
4,China,United States,International
5,Brazil,Brazil,Domestic
6,Ivory Coast,United Kingdom,International
7,Vietnam,Brazil,International
8,Palestinian Territory,Italy,International
9,Thailand,Canada,International


In [224]:
df.head(5)

,Passenger_ID,First_Name,Last_Name,Gender,Age,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Departure_Date,Arrival_Airport,Pilot_Name,Flight_Status,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek,Month_Name,Age_Group,Passenger_Full_Name,Is_Flight_Delayed,Delay_Cause,Delay_Severity,Operational_Risk,Pilot_Workload,Monthly_Traffic,Weekly_Traffic,Airport_Congestion,Airport_Location,Flight_Type
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,2022-06-28,CXF,Fransisco Hazeldine,On Time,2022,6,28,Tuesday,Jun,Senior Citizen,Edithe Leggis,0,No Delay,No Impact,Low Risk,1,4872,8386,11,"Coldfoot Airport, United States",International
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,2022-12-26,YCO,Marla Parsonage,On Time,2022,12,26,Monday,Dec,Senior Citizen,Elwood Catt,0,No Delay,No Impact,Low Risk,1,4732,8758,9,"Kugluktuk Airport, Canada",International
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,2022-01-18,GNB,Rhonda Amber,On Time,2022,1,18,Tuesday,Jan,Senior Citizen,Darby Felgate,0,No Delay,No Impact,Low Risk,1,5217,8386,15,"Grenoble-Isère Airport, France",International
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,2022-09-16,YND,Kacie Commucci,Delayed,2022,9,16,Friday,Sep,Senior Citizen,Dominica Pyle,1,Operational / Technical,Operational Impact,Medium Risk,1,4897,8462,7,"Ottawa / Gatineau Airport, Canada",International
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2022-02-25,SEE,Ebonee Tree,On Time,2022,2,25,Friday,Feb,Youth,Bay Pencost,0,No Delay,No Impact,Low Risk,1,4398,8462,11,"Gillespie Field, United States",International


In [225]:
print("Final dataset shape:", df.shape)
print("Final columns:")
df.columns.tolist()


Final dataset shape: (98619, 32)
Final columns:


['Passenger_ID',
 'First_Name',
 'Last_Name',
 'Gender',
 'Age',
 'Nationality',
 'Airport_Name',
 'Airport_Country_Code',
 'Country_Name',
 'Airport_Continent',
 'Continents',
 'Departure_Date',
 'Arrival_Airport',
 'Pilot_Name',
 'Flight_Status',
 'Departure_Year',
 'Departure_Month',
 'Departure_Day',
 'Departure_DayOfWeek',
 'Month_Name',
 'Age_Group',
 'Passenger_Full_Name',
 'Is_Flight_Delayed',
 'Delay_Cause',
 'Delay_Severity',
 'Operational_Risk',
 'Pilot_Workload',
 'Monthly_Traffic',
 'Weekly_Traffic',
 'Airport_Congestion',
 'Airport_Location',
 'Flight_Type']

Reorder Columns into a clean, logical sequence

In [226]:


final_column_order = [
"Passenger_ID",
"First_Name",
"Last_Name",
"Passenger_Full_Name",
"Gender",
"Age",
"Age_Group",
"Nationality",

"Airport_Name",
"Airport_Country_Code",
"Country_Name",
"Airport_Continent",
"Continents",
"Airport_Location",
"Airport_Congestion",

"Departure_Date",
"Departure_Year",
"Departure_Month",
"Departure_Day",
"Departure_DayOfWeek",
'Month_Name',
"Monthly_Traffic",
"Weekly_Traffic",

"Arrival_Airport",
"Pilot_Name",
"Pilot_Workload",

"Flight_Status",
"Is_Flight_Delayed",
"Delay_Cause",
"Delay_Severity",
"Operational_Risk",

"Flight_Type"
]

df = df[final_column_order]

print("Columns reordered successfully")
df.head()


Columns reordered successfully


,Passenger_ID,First_Name,Last_Name,Passenger_Full_Name,Gender,Age,Age_Group,Nationality,Airport_Name,Airport_Country_Code,Country_Name,Airport_Continent,Continents,Airport_Location,Airport_Congestion,Departure_Date,Departure_Year,Departure_Month,Departure_Day,Departure_DayOfWeek,Month_Name,Monthly_Traffic,Weekly_Traffic,Arrival_Airport,Pilot_Name,Pilot_Workload,Flight_Status,Is_Flight_Delayed,Delay_Cause,Delay_Severity,Operational_Risk,Flight_Type
0,ABVWIg,Edithe,Leggis,Edithe Leggis,Female,62,Senior Citizen,Japan,Coldfoot Airport,US,United States,NAM,North America,"Coldfoot Airport, United States",11,2022-06-28,2022,6,28,Tuesday,Jun,4872,8386,CXF,Fransisco Hazeldine,1,On Time,0,No Delay,No Impact,Low Risk,International
1,jkXXAX,Elwood,Catt,Elwood Catt,Male,62,Senior Citizen,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,"Kugluktuk Airport, Canada",9,2022-12-26,2022,12,26,Monday,Dec,4732,8758,YCO,Marla Parsonage,1,On Time,0,No Delay,No Impact,Low Risk,International
2,CdUz2g,Darby,Felgate,Darby Felgate,Male,67,Senior Citizen,Russia,Grenoble-Isère Airport,FR,France,Europe,Europe,"Grenoble-Isère Airport, France",15,2022-01-18,2022,1,18,Tuesday,Jan,5217,8386,GNB,Rhonda Amber,1,On Time,0,No Delay,No Impact,Low Risk,International
3,BRS38V,Dominica,Pyle,Dominica Pyle,Female,71,Senior Citizen,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,"Ottawa / Gatineau Airport, Canada",7,2022-09-16,2022,9,16,Friday,Sep,4897,8462,YND,Kacie Commucci,1,Delayed,1,Operational / Technical,Operational Impact,Medium Risk,International
4,9kvTLo,Bay,Pencost,Bay Pencost,Male,21,Youth,China,Gillespie Field,US,United States,NAM,North America,"Gillespie Field, United States",11,2022-02-25,2022,2,25,Friday,Feb,4398,8462,SEE,Ebonee Tree,1,On Time,0,No Delay,No Impact,Low Risk,International


Saving Processed Data file

In [227]:
OUT_PATH = "../data/processed/week2_processed.csv"

df.to_csv(OUT_PATH, index=False)

print("Week 2 processed file saved successfully")
print("Saved at:", OUT_PATH)


Week 2 processed file saved successfully
Saved at: ../data/processed/week2_processed.csv


### Week 2 Summary Insights – Feature Engineering & Transformation

Week 2 focused on transforming cleaned aviation data into meaningful
analytical features to support operational and delay analysis.


### Outcomes

- Converted raw aviation records into analytical intelligence.
- Enabled delay prediction and operational performance analysis.
- Established foundational metrics used in Week 3 exploration
  and Week 4 delay analytics.

The engineered dataset significantly enhanced analytical depth
and supported managerial decision-making insights.